In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [2]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *
from kret_optuna._core.optuna_nb_imports import *
from kret_polars._core.polars_nb_imports import *
from kret_rosetta._core.rosetta_nb_imports import *
from kret_sklearn._core.sklearn_nb_imports import *
from kret_torch_utils._core.torch_nb_imports import *
from kret_tqdm._core.tqdm_nb_imports import *
from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

Loaded environment variables from /Users/Akseldkw/coding/Columbia/NBA-Timeout-Impact/.env
[kret_lgbm._core.lgbm_nb_imports] Imported kret_lgbm._core.lgbm_nb_imports in 2.3356 seconds
[kret_lightning._core.lightning_nb_imports] Imported kret_lightning._core.lightning_nb_imports in 4.6554 seconds
[kret_matplotlib._core.mpl_nb_imports] Imported kret_matplotlib._core.mpl_nb_imports in 0.0305 seconds
[kret_np_pd._core.np_pd_nb_imports] Imported kret_np_pd._core.np_pd_nb_imports in 0.0006 seconds
[kret_optuna._core.optuna_nb_imports] Imported kret_optuna._core.optuna_nb_imports in 0.0006 seconds
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0975 seconds
[kret_rosetta._core.rosetta_nb_imports] Imported kret_rosetta._core.rosetta_nb_imports in 0.0000 seconds
[kret_sklearn._core.sklearn_nb_imports] Imported kret_sklearn._core.sklearn_nb_imports in 0.1819 seconds
[kret_torch_utils._core.torch_nb_imports] Imported kret_torch_utils._core.torch_nb_imports i

In [ ]:
from nba_timeout_impact.load_data_utils import NBADataLoader
from nba_timeout_impact.clean_pipeline import NBAStatsV3CleanPipeline
from nba_timeout_impact.constants import NBAConstants
from nba_timeout_impact.nba_dataset import NBADataset

In [4]:
UKS_CONSTANTS.DATA_DIR

PosixPath('/Users/Akseldkw/coding/data_kretsinger')

In [5]:
NBA_DATA_DIR = UKS_CONSTANTS.DATA_DIR / "NBA"
os.makedirs(NBA_DATA_DIR, exist_ok=True)

# Load Data

In [ ]:
# nba_statsv3 = NBADataLoader.load_seasons(data_type="nbastatsv3", playoffs=False, seasons=None)
nba_statsv3 = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_regular_season.parquet")
nba_statsv3["IsPlayoff"] = False

In [ ]:
# nba_statsv3_po = NBADataLoader.load_seasons(data_type="nbastatsv3", playoffs=True, seasons=None)
nba_statsv3_po = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_playoffs.parquet")
nba_statsv3_po["IsPlayoff"] = True

In [55]:
nba_statsv3_all = pd.concat([nba_statsv3, nba_statsv3_po], ignore_index=True)

In [56]:
nba_statsv3_all.to_parquet(NBA_DATA_DIR / "nba_statsv3.parquet")

In [ ]:
# early = NBADataLoader.load_game_dates_from_api(seasons=[1996, 1997, 1998, 1999])
# early_po = NBADataLoader.load_game_dates_from_api(seasons=[1996, 1997, 1998, 1999], playoffs=True)
# dates_nba = NBADataLoader.load_game_dates()
# dates_nba_po = NBADataLoader.load_game_dates(playoffs=True)

nba_game_dates = pd.read_parquet(NBA_DATA_DIR / "nba_game_dates.parquet")
dates_all = nba_game_dates

In [52]:
game_date_map = dates_all.set_index("GAME_ID")["game_date"]
nba_statsv3["game_date"] = nba_statsv3["gameId"].map(game_date_map)

# Implementation

In [ ]:
# nba_statsv3.to_parquet(NBA_DATA_DIR / "nba_statsv3_regular_season.parquet")

In [ ]:
# nba_statsv3_po.to_parquet(NBA_DATA_DIR / "nba_statsv3_playoffs.parquet")

In [ ]:
# reload = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_regular_season.parquet")

# Pipeline

In [86]:
from nba_timeout_impact.clean_pipeline import SortByDate, CategoricalConversion

In [87]:
pipeline = NBAStatsV3CleanPipeline(NBA_DATA_DIR)
clean = pipeline.run()

In [88]:
UKS_NP_PD.dtt([clean], how="head", show_dims=True, n=20)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,shotValue,IsPlayoff,seconds_remaining,seconds_elapsed,game_date,game_date_ffill
,int64,str,int64,category,category,int64,category,category,int64,int64,int64,category,int64,float64,float64,int64,category,str,category,category,int64,int64,int64,float64,bool,float64,float64,datetime64[s],datetime64[s]
0,1,PT12M00.00S,1,0,NaN,0,NaN,NaN,0,0,0,NaN,0,0.000,0.000,0,NaN,Start of 1st Period (11:15 PM EST),period,start,0,1,29600001,NaN,False,720.000,0.000,1996-11-01,1996-11-01
1,2,PT12M00.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,0,0,NaN,0,0.000,0.000,0,h,Jump Ball Ellison vs. Longley: Tip to Harper,Jump Ball,NaN,0,2,29600001,NaN,False,720.000,0.000,1996-11-01,1996-11-01
2,4,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,0,0,Made,1,0.000,2.000,2,v,Rodman Layup (2 PTS) (Longley 1 AST),Made Shot,Layup Shot,0,3,29600001,NaN,False,699.000,21.000,1996-11-01,1996-11-01
3,5,PT11M39.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,0,0,NaN,0,0.000,0.000,0,h,Ellison S.FOUL (P1.T1),Foul,Shooting,0,4,29600001,NaN,False,699.000,21.000,1996-11-01,1996-11-01
4,6,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,0,0,NaN,0,0.000,3.000,3,v,Rodman Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,0,5,29600001,NaN,False,699.000,21.000,1996-11-01,1996-11-01
5,7,PT11M06.00S,1,1610612738,BOS,133,Wesley,D. Wesley,163,122,20,Made,1,2.000,3.000,5,h,Wesley 20' Jump Shot (2 PTS),Made Shot,Jump Shot,0,6,29600001,NaN,False,666.000,54.000,1996-11-01,1996-11-01
6,43,PT11M06.00S,1,1610612741,CHI,893,Jordan,M. Jordan,45,148,15,Missed,1,0.000,0.000,0,v,MISS Jordan 15' Jump Shot,Missed Shot,Jump Shot,0,7,29600001,NaN,False,666.000,54.000,1996-11-01,1996-11-01
7,44,PT11M06.00S,1,1610612738,BOS,133,Wesley,D. Wesley,0,0,0,NaN,0,0.000,0.000,0,h,Wesley REBOUND (Off:0 Def:1),Rebound,Unknown,0,8,29600001,NaN,False,666.000,54.000,1996-11-01,1996-11-01
8,8,PT11M02.00S,1,1610612738,BOS,677,Williams,E. Williams,0,0,0,Made,1,4.000,3.000,7,h,Williams Driving Layup (2 PTS) (Wesley 1 AST),Made Shot,Driving Layup Shot,0,9,29600001,NaN,False,662.000,58.000,1996-11-01,1996-11-01


In [89]:
UKS_NP_PD.count_unique(clean)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,shotValue,IsPlayoff,seconds_remaining,seconds_elapsed,game_date,game_date_ffill
,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64
0,998,1261,8,31,36,3638,1887,2597,539,929,93,2,2,173,173,342,2,3784108,14,185,2,745,37514,3,2,1261,1801,5869,5869


In [92]:
type(pd.to_datetime(clean.game_date))

pandas.Series

In [90]:
pipeline.save(clean, NBA_DATA_DIR / "nba_statsv3_playoffs_clean.parquet")

Saved 18,049,803 rows x 29 cols -> /Users/Akseldkw/coding/data_kretsinger/NBA/nba_statsv3_playoffs_clean.parquet


PosixPath('/Users/Akseldkw/coding/data_kretsinger/NBA/nba_statsv3_playoffs_clean.parquet')

In [45]:
reload = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_playoffs_clean.parquet")

In [ ]:
sort = SortByDate()

In [78]:
sort.fit(reload)
out = sort.transform(reload)

In [79]:
dtt([reload, out], num_cols=1, how="head", n=20, max_col_width=250)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,shotValue,seconds_remaining,seconds_elapsed,game_date,game_date_ffill
,int64,str,int64,int64,category,int64,category,category,int64,int64,int64,str,int64,float64,float64,int64,str,str,category,category,int64,int64,int64,float64,float64,float64,datetime64[ms],datetime64[ms]
0,1,PT12M00.00S,1,0,NaN,0,NaN,NaN,0,0,0,NaN,0,0.000,0.000,0,NaN,Start of 1st Period (11:15 PM EST),period,start,0,1,29600001,NaN,720.000,0.000,1996-11-01,1996-11-01
1,2,PT12M00.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,0,0,NaN,0,0.000,0.000,0,h,Jump Ball Ellison vs. Longley: Tip to Harper,Jump Ball,NaN,0,2,29600001,NaN,720.000,0.000,1996-11-01,1996-11-01
2,4,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,0,0,Made,1,0.000,2.000,2,v,Rodman Layup (2 PTS) (Longley 1 AST),Made Shot,Layup Shot,0,3,29600001,NaN,699.000,21.000,1996-11-01,1996-11-01
3,5,PT11M39.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,0,0,NaN,0,0.000,0.000,0,h,Ellison S.FOUL (P1.T1),Foul,Shooting,0,4,29600001,NaN,699.000,21.000,1996-11-01,1996-11-01
4,6,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,0,0,NaN,0,0.000,3.000,3,v,Rodman Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,0,5,29600001,NaN,699.000,21.000,1996-11-01,1996-11-01
5,7,PT11M06.00S,1,1610612738,BOS,133,Wesley,D. Wesley,163,122,20,Made,1,2.000,3.000,5,h,Wesley 20' Jump Shot (2 PTS),Made Shot,Jump Shot,0,6,29600001,NaN,666.000,54.000,1996-11-01,1996-11-01
6,43,PT11M06.00S,1,1610612741,CHI,893,Jordan,M. Jordan,45,148,15,Missed,1,0.000,0.000,0,v,MISS Jordan 15' Jump Shot,Missed Shot,Jump Shot,0,7,29600001,NaN,666.000,54.000,1996-11-01,1996-11-01
7,44,PT11M06.00S,1,1610612738,BOS,133,Wesley,D. Wesley,0,0,0,NaN,0,0.000,0.000,0,h,Wesley REBOUND (Off:0 Def:1),Rebound,Unknown,0,8,29600001,NaN,666.000,54.000,1996-11-01,1996-11-01
8,8,PT11M02.00S,1,1610612738,BOS,677,Williams,E. Williams,0,0,0,Made,1,4.000,3.000,7,h,Williams Driving Layup (2 PTS) (Wesley 1 AST),Made Shot,Driving Layup Shot,0,9,29600001,NaN,662.000,58.000,1996-11-01,1996-11-01


In [81]:
is_sorted_per_group = reload.groupby("gameId")["actionId"].apply(lambda s: s.is_monotonic_increasing)
is_sorted_per_group2 = out.groupby("gameId")["actionId"].apply(lambda s: s.is_monotonic_increasing)

In [82]:
is_sorted_per_group.mean(), is_sorted_per_group2.mean()

(np.float64(1.0), np.float64(1.0))